# Swing+ Results

In [ ]:
# Setup: paths + imports shared by every cell below. The leaderboards are rendered by R
# (see the "Aesthetic leaderboards" section); the validation cells build their own frames.
from pathlib import Path
import pandas as pd
import dataframe_image as dfi

ROOT = next(p for p in [Path.cwd(), Path.cwd().parent] if (p / 'data').exists())
DATA = ROOT / 'data'
PLOTS = ROOT / 'results' / 'plots'
PLOTS.mkdir(parents=True, exist_ok=True)


def plot_path(name):
    """Route a figure filename to its results/plots/<category> subfolder (kept mirrored in the R
    generator and cluster_results.ipynb)."""
    if name.startswith('swing_cards'):
        sub = 'swing_cards'
    elif name.startswith('usage_heatmap'):
        sub = 'usage_heatmap'
    elif name.startswith('repertoire'):
        sub = 'repertoire'
    elif name.startswith(('swingplus_leaderboard', 'swingplus_bottom', 'swingplus_by_cluster', 'shape_breakdown')):
        sub = 'swing_plus'
    else:
        sub = 'predictiveness'
    d = PLOTS / sub
    d.mkdir(parents=True, exist_ok=True)
    return d / name

## Aesthetic leaderboards (R · gt + headshots)

Swing+, Repertoire+, and Swing+-by-shape leaderboards rendered with **`gt` + `mlbplotR`** (MLBAM headshots, 538 theme) via `src/leaderboard_table.R`, reading the same `data/*.parquet`. **Top 25 and Bottom 25** for each. The value column is colored **red (high) → blue (low)** with the domain fixed to the *full* qualified pool, so the Top 25 reads all-red and the Bottom 25 all-blue. The by-shape table reports each shape's **% of the hitter's swings** (usage share). PNGs saved to `results/plots/*_gt.png`.

In [ ]:
# Aesthetic leaderboards via R (gt + mlbplotR MLBAM headshots). Runs src/leaderboard_table.R, which
# reads the same data/*.parquet and writes results/plots/{swing_plus,repertoire}/*.png, then displays
# them inline. Value color is fixed to the FULL qualified pool (red = high, blue = low), so Top 25
# reads all-red / Bottom 25 all-blue. Rscript not on PATH here -> fall back to the R 4.6.0 install.
import subprocess, shutil
from IPython.display import Image, display

RSCRIPT = shutil.which('Rscript') or r'C:\Users\theo.an-yeung\AppData\Local\Programs\R\R-4.6.0\bin\Rscript.exe'
subprocess.run([RSCRIPT, 'src/leaderboard_table.R'], cwd=str(ROOT), check=True)
gt_pngs = [
    'swingplus_leaderboard_gt.png', 'swingplus_bottom_gt.png',          # Swing+ (batter): top / bottom
    'repertoire_leaderboard_gt.png', 'repertoire_bottom_gt.png',        # Repertoire+ (unit): top / bottom
    'swingplus_by_cluster_gt.png', 'swingplus_by_cluster_bottom_gt.png',# Swing+ by shape: top / bottom
]
for png in gt_pngs:
    display(Image(filename=str(plot_path(png))))

### Per-hitter breakdown

`shape_breakdown("name")` renders a single hitter's swing shapes (case-insensitive substring; a switch hitter returns both stances) ranked by Swing+, via the same R script in **breakdown mode** — headshot, archetype·situation, usage %, and Swing+. Color is the **league scale** (all qualified shapes), so a below-average hitter's shapes read pale/blue. Writes `results/plots/shape_breakdown_<name>_gt.png`.

In [ ]:
# Per-hitter breakdown: one batter's swing shapes ranked by Swing+, colored on the league scale,
# rendered by the same R script in breakdown mode (headshot + usage %). `name` is a case-insensitive
# substring, so a switch hitter returns both stances (e.g. "Raleigh" -> Cal Raleigh L and R).
import re

def shape_breakdown(name):
    subprocess.run([RSCRIPT, 'src/leaderboard_table.R', name], cwd=str(ROOT), check=True)
    slug = re.sub(r'(^_|_$)', '', re.sub(r'[^a-z0-9]+', '_', name.lower()))
    return Image(filename=str(plot_path(f'shape_breakdown_{slug}_gt.png')))

shape_breakdown('Arraez')

### Repertoire+ vs Swing+

Width and quality are independent axes — a wide repertoire says nothing about whether the swings are good. Scatter to confirm there's no clear relationship.

In [ ]:
# Repertoire+ vs Swing+ — two independent axes (width vs quality). Expect ~no relationship.
import numpy as np
import matplotlib.pyplot as plt

rep = pd.read_parquet(DATA / 'repertoire_scores.parquet',
                      columns=['batter_id', 'batter_stand', 'repertoire_plus'])
ca = pd.read_parquet(DATA / 'cluster_assignments.parquet',
                     columns=['play_id', 'batter_id', 'batter_stand'])
sw = pd.read_parquet(DATA / 'xrv_swings.parquet', columns=['play_id', 'game_year', 'xrv_grade'])
sp = (ca.merge(sw, on='play_id').query('game_year in [2024, 2025]')
        .groupby(['batter_id', 'batter_stand'])['xrv_grade'].mean().rename('swing_plus').reset_index())
d = rep.merge(sp, on=['batter_id', 'batter_stand']).dropna(subset=['repertoire_plus', 'swing_plus'])
r = np.corrcoef(d['repertoire_plus'], d['swing_plus'])[0, 1]
b1, b0 = np.polyfit(d['repertoire_plus'], d['swing_plus'], 1)

FT = ['fivethirtyeight', {'figure.facecolor': 'white', 'axes.facecolor': 'white',
                          'savefig.facecolor': 'white', 'grid.color': '#cbcbcb'}]
with plt.style.context(FT):
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(d['repertoire_plus'], d['swing_plus'], s=18, alpha=.45, color='#30a2da', edgecolor='none')
    xs = np.array([d['repertoire_plus'].min(), d['repertoire_plus'].max()])
    ax.plot(xs, b0 + b1 * xs, color='#fc4f30', lw=1.5, label=f'OLS fit (slope {b1:.2f})')
    ax.set_xlabel('Repertoire+  (repertoire width)')
    ax.set_ylabel('Swing+  (swing quality)')
    ax.set_title(f'Repertoire+ vs Swing+ — no clear relationship  (r = {r:.2f}, n = {len(d)})', fontsize=12)
    ax.legend()
    fig.tight_layout()
    fig.savefig(plot_path('repertoire_vs_swingplus.png'), dpi=130)
    plt.show()
print(f'r = {r:.3f}, n = {len(d)}')

## Validation

Four checks on the bespoke xRV / Swing+ pipeline:

1. **Year over year** — per-season agreement of predicted (`xrv`) and tables-realized (`realized_rv`) run value against the `delta_run_exp` anchor (models train on 2024–25; **2026 is held out**), then Swing+ reliability across consecutive seasons (is it a stable batter skill?).
2. **Expected vs actual** — batter-level scatter of mean predicted xRV against mean actual `delta_run_exp`.
3. **Feature importance** — gain per feature for each of the three models (`p_bip`, `p_foul`, `v_bip`), loaded from `data/xrv_models/` (produced by `src/xRV_model.py`).
4. **Predictive validity** — faceted scatter of Swing+ against real hitting outcomes (BA, OPS, wRC+, wOBA, WAR) from `season_stats_hitting`, one panel per metric with its Pearson r. *OPS+ is not stored in `mlb_db`; wRC+ is the park/league-adjusted analog and stands in for it.*

In [ ]:
# ---- Validation setup + year-over-year model agreement ----
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error

# Full per-swing xRV table: predicted (xrv), tables-realized (realized_rv), and the actual anchor (delta_run_exp)
val = pd.read_parquet(DATA / 'xrv_swings.parquet',
                      columns=['batter_id', 'batter_stand', 'game_year', 'balls', 'strikes',
                               'xrv', 'realized_rv', 'delta_run_exp', 'xrv_grade'])
print(f'{len(val):,} swings | seasons {sorted(val.game_year.unique())}')

# (a) Per-season agreement with delta_run_exp. Models train on 2024-25; 2026 is the held-out test season.
m = val.dropna(subset=['delta_run_exp'])
per_season = (m.groupby('game_year')
              .apply(lambda g: pd.Series({
                  'swings': len(g),
                  'corr(xRV, actual)': np.corrcoef(g['xrv'], g['delta_run_exp'])[0, 1],
                  'corr(realized, actual)': np.corrcoef(g['realized_rv'], g['delta_run_exp'])[0, 1],
                  'rmse(realized, actual)': root_mean_squared_error(g['delta_run_exp'], g['realized_rv']),
                  'mean xRV': g['xrv'].mean()}), include_groups=False)
              .round(4))
per_season['swings'] = per_season['swings'].astype(int)
dfi.export(per_season, str(plot_path('xrv_per_season_validation.png')), table_conversion='matplotlib')
print('wrote', plot_path('xrv_per_season_validation.png'), '  (2026 = held-out test season)')
per_season

In [ ]:
# (b) Year-over-year reliability: is Swing+ a stable batter skill? Correlate each batter's Swing+
# across consecutive seasons (batters with >= 200 tracked swings in both years).
MIN_SW = 200
sp = (val.groupby(['batter_id', 'game_year'])
        .agg(swings=('xrv_grade', 'size'), swing_plus=('xrv_grade', 'mean'))
        .reset_index().query('swings >= @MIN_SW'))
wide = sp.pivot(index='batter_id', columns='game_year', values='swing_plus')
rel = []
for a, b in [(2024, 2025), (2025, 2026)]:
    d = wide[[a, b]].dropna()
    rel.append({'season pair': f'{a} -> {b}', 'batters': len(d), 'Swing+ corr': np.corrcoef(d[a], d[b])[0, 1]})
yoy_reliability = pd.DataFrame(rel).set_index('season pair').round(3)
dfi.export(yoy_reliability, str(plot_path('swingplus_yoy_reliability.png')), table_conversion='matplotlib')
print('wrote', plot_path('swingplus_yoy_reliability.png'))
yoy_reliability

In [ ]:
# ---- Feature importance per model (gain), models loaded from src/xRV_model.py's outputs ----
# Three separate models share the same 11 predictors; we plot each one's gain split.
import sys
import xgboost as xgb
sys.path.insert(0, str(ROOT / 'src'))
from xRV_model import FEATURES, SHAPE_FEATURES, MODEL_DIR   # 11 shared predictors + saved-model dir

def gain(name):
    b = xgb.Booster()
    b.load_model(str(MODEL_DIR / f'{name}.json'))
    s = pd.Series(b.get_score(importance_type='gain')).reindex(FEATURES).fillna(0.0)
    return 100 * s / s.sum()   # % of total gain

MODELS = {'p_bip': 'p_bip  P(in play)',
          'p_foul': 'p_foul  P(foul | not BIP)',
          'v_bip': 'v_bip  E[batted-ball RV]'}
imp = pd.DataFrame({n: gain(n) for n in MODELS})

FT = ['fivethirtyeight', {'figure.facecolor': 'white', 'axes.facecolor': 'white',
                          'savefig.facecolor': 'white', 'grid.color': '#cbcbcb'}]
with plt.style.context(FT):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
    for ax, name in zip(axes, MODELS):
        d = imp[name].sort_values()
        colors = ['#fc4f30' if f in SHAPE_FEATURES else '#30a2da' for f in d.index]  # red = shape, blue = context
        ax.barh(d.index, d.values, color=colors)
        ax.set_title(MODELS[name], fontsize=11)
        ax.set_xlabel('% of total gain')
    fig.suptitle('xRV feature importance by model  (red = 5 swing-shape features, blue = context)', fontsize=13)
    fig.tight_layout()
    fig.savefig(plot_path('xrv_feature_importance.png'), dpi=130)
print('wrote', plot_path('xrv_feature_importance.png'))
imp.round(1)

In [ ]:
# ---- Predictive validity: does Swing+ track real hitting outcomes? ----
# OPS+ is NOT stored in mlb_db. wRC+ (wrc_plus) is the park/league-adjusted analog (100 = avg), so it
# stands in for OPS+ here. We pull season_stats_hitting (MLB, regular season) and scatter Swing+
# against each outcome per (batter, season), one faceted panel each with its Pearson r + OLS fit.
import os, re, warnings
import mysql.connector
warnings.filterwarnings('ignore', message='pandas only supports SQLAlchemy')

def get_secret(name):
    v = os.environ.get(name)
    if v:
        return v
    ef = Path.home() / '.claude' / '.env'
    if ef.exists():
        for line in ef.read_text(encoding='utf-8', errors='ignore').splitlines():
            m = re.match(rf'^\s*{re.escape(name)}\s*=\s*(.+)$', line)
            if m:
                return m.group(1).strip().strip('"').strip("'")
    return None

cn = mysql.connector.connect(host=get_secret('BIOMECH_DB_HOST'), port=int(get_secret('BIOMECH_DB_PORT') or 3306),
                             user=get_secret('BIOMECH_DB_USER'), password=get_secret('BIOMECH_DB_PASS'), database='mlb_db')
stats = pd.read_sql("""SELECT season AS game_year, mlbam_id AS batter_id, pa,
    ba, ops, wrc_plus, woba, war
    FROM season_stats_hitting
    WHERE level_id = 1 AND game_type = 'R' AND season IN (2024, 2025, 2026)""", cn)
cn.close()

MIN_SW, MIN_PA = 150, 200
bs = (val.groupby(['batter_id', 'game_year'])
        .agg(swings=('xrv_grade', 'size'), swing_plus=('xrv_grade', 'mean'))
        .reset_index().query('swings >= @MIN_SW'))
mrg = bs.merge(stats, on=['batter_id', 'game_year'], how='inner').query('pa >= @MIN_PA')
print(f'Batter-seasons: {len(mrg)} (>= {MIN_SW} tracked swings & >= {MIN_PA} PA), MLB regular season 2024-26')

# wRC+ is the OPS+ analog (OPS+ not in mlb_db). One panel per outcome, Swing+ on x.
targets = {'ba': 'BA', 'ops': 'OPS', 'wrc_plus': 'wRC+', 'woba': 'wOBA', 'war': 'WAR'}
FT = ['fivethirtyeight', {'figure.facecolor': 'white', 'axes.facecolor': 'white',
                          'savefig.facecolor': 'white', 'grid.color': '#cbcbcb'}]
with plt.style.context(FT):
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    for ax, (col, lab) in zip(axes.flat, targets.items()):
        d = mrg[['swing_plus', col]].dropna()
        r = d['swing_plus'].corr(d[col])
        slope, intercept = np.polyfit(d['swing_plus'], d[col], 1)
        xs = np.array([d['swing_plus'].min(), d['swing_plus'].max()])
        ax.scatter(d['swing_plus'], d[col], s=16, alpha=.45, color='#30a2da', edgecolor='none')
        ax.plot(xs, intercept + slope * xs, color='#fc4f30', lw=1.8)
        ax.set_title(f'{lab}   (r = {r:.3f}, n = {len(d)})', fontsize=11)
        ax.set_xlabel('Swing+')
        ax.set_ylabel(lab)
    axes.flat[-1].axis('off')   # 5 metrics in a 2x3 grid -> hide the empty 6th panel
    fig.suptitle('Swing+ vs real hitting outcomes (batter-season, MLB regular season 2024-26)', fontsize=13)
    fig.tight_layout()
    fig.savefig(plot_path('swingplus_predictive_scatter.png'), dpi=130)
    plt.show()
print('wrote', plot_path('swingplus_predictive_scatter.png'))